# QA Analytics & Data Quality Dashboard
## Análisis Completo de Evaluaciones QA — 2026

---

### Propósito de este notebook

Este notebook documenta el proceso completo de análisis de calidad para el dataset de evaluaciones QA del año 2026.
Cada sección explica:
- **Qué** estamos haciendo
- **Por qué** lo hacemos
- **Qué significa** el resultado
- **Qué decisión de negocio** puede apoyarse con ese análisis

### Flujo del análisis
```
1. Setup & Carga de datos
2. Data Quality Analysis
3. Data Cleaning Pipeline
4. KPI Calculation
5. Team Performance
6. Process Performance
7. Agent Performance (Top/Bottom)
8. Trend Analysis
9. Correlation Analysis
10. Visualizations
11. Executive Insights
```

---
## 1. Setup & Carga de Datos

**Qué hacemos:** Importar las librerías necesarias y cargar el dataset crudo desde CSV.

**Por qué:** Establecer el entorno de trabajo antes de cualquier análisis. Separamos la carga del análisis para poder reutilizar el dataset sin re-leer el archivo múltiples veces.

**Buena práctica:** Definir todas las rutas como variables al inicio del notebook para facilitar el mantenimiento.

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Agregar src/ al path para importar módulos del proyecto
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

# Importar módulos del proyecto
from data_quality   import run_data_quality_report, results_to_dataframe
from data_cleaning  import run_cleaning_pipeline, get_clean_dataset
from kpis           import (calculate_all_kpis, print_kpi_summary,
                            kpis_to_dataframe, qa_category_distribution)
from analysis       import (team_performance, process_performance,
                            agent_performance, top_agents, bottom_agents,
                            daily_trend, weekly_trend, monthly_trend,
                            correlation_analysis, generate_executive_insights)
from visualizations import generate_all_visualizations

# Rutas
DATA_PATH   = ROOT / 'Data'  / 'qa_evaluations_2026.csv'
OUTPUT_DIR  = ROOT / 'Output'
VIZ_DIR     = ROOT / 'visualizations'

# Configuración de visualizaciones en notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print('✅ Setup completado')
print(f'   Dataset path: {DATA_PATH}')
print(f'   Python: {sys.version.split()[0]}')
print(f'   Pandas: {pd.__version__}')
print(f'   NumPy:  {np.__version__}')
print(f'   Matplotlib: {matplotlib.__version__}')

In [ ]:
# Cargar el dataset crudo
# NUNCA modificar df_raw — es nuestra fuente de verdad
df_raw = pd.read_csv(DATA_PATH)

print(f'Dataset cargado: {len(df_raw):,} registros × {len(df_raw.columns)} columnas')
df_raw.head()

---
## 2. Data Quality Analysis

**Qué hacemos:** Ejecutar 14 validaciones automáticas sobre el dataset crudo.

**Por qué:** Un análisis confiable requiere datos confiables. Antes de calcular cualquier KPI, necesitamos saber:
- ¿Hay duplicados que inflen los conteos?
- ¿Hay valores fuera de rango que distorsionen los promedios?
- ¿Hay nulos que excluyan registros silenciosamente?

**Decisión de negocio:** El Data Quality Score determina si podemos confiar en las conclusiones del análisis. Un score bajo requiere escalamiento al equipo de sistemas antes de reportar KPIs.

In [ ]:
# Ejecutar el reporte completo de calidad de datos
dq_results = run_data_quality_report(df_raw)

In [ ]:
# Convertir a DataFrame para visualización tabular
dq_df = results_to_dataframe(dq_results)
dq_df[dq_df['Category'].isin(['Dimensions', 'Duplicates', 'Missing Values', 'Validation', 'Overall'])]

### Interpretación del Data Quality Score

El **Data Quality Score** mide qué porcentaje de los registros son completamente válidos:

```
DQ Score = max(0, 1 - total_issues / total_records) × 100
```

| Score | Nivel | Acción |
|-------|-------|--------|
| >= 98% | Alta calidad | Proceder con análisis |
| 95–97% | Buena calidad | Limpieza menor |
| 90–94% | Aceptable | Limpieza moderada |
| < 90% | Baja calidad | Revisar con sistemas |

---
## 3. Data Cleaning Pipeline

**Qué hacemos:** Ejecutar el pipeline de 6 pasos que transforma el dataset crudo en un dataset limpio y enriquecido.

**Por qué:** Los datos crudos raramente están listos para análisis directo. El pipeline garantiza:
1. Reproducibilidad — el mismo input siempre produce el mismo output
2. Trazabilidad — los registros inválidos se marcan, no se eliminan
3. Enriquecimiento — columnas derivadas facilitan el análisis posterior

**Decisión de negocio:** La decisión de imputar CSAT con mediana grupal (vs eliminar los registros con nulos) preserva el volumen del análisis sin introducir sesgo de media en una escala ordinal.

In [ ]:
# Ejecutar el pipeline de limpieza
# El resultado se guarda en output/qa_clean.csv
CLEAN_PATH = OUTPUT_DIR / 'qa_clean.csv'
df_clean_full = run_cleaning_pipeline(df_raw, output_path=CLEAN_PATH)

# df es el dataset de trabajo — solo registros válidos
df = get_clean_dataset(df_clean_full)

In [ ]:
# Ver las nuevas columnas derivadas
print('Columnas del dataset limpio:')
for col in df.columns:
    print(f'  {col:<30} {df[col].dtype}')
print(f'\nTotal registros válidos: {len(df):,}')

In [ ]:
# Distribución de QA Categories — vista rápida
print('Distribución de QA Categories:')
cat_counts = df['QA_Category'].value_counts()
for cat, cnt in cat_counts.items():
    pct = cnt / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cat:<15} {bar:<25} {cnt:>6,} ({pct:.1f}%)')

---
## 4. KPI Calculation

**Qué hacemos:** Calcular los 8 KPIs principales y compararlos con sus targets operacionales.

**Por qué:** Los KPIs son el lenguaje del negocio. Transforman miles de evaluaciones individuales en números accionables que los líderes pueden usar para tomar decisiones.

**Decisión de negocio:** Para Resolution Time no establecemos un target arbitrario; analizamos la tendencia. Un tiempo promedio de 9 minutos no es bueno o malo sin contexto de industria o histórico.

In [ ]:
# Calcular todos los KPIs
kpis = calculate_all_kpis(df)
print_kpi_summary(kpis)

In [ ]:
# KPIs en tabla para exportación
kpi_df = kpis_to_dataframe(kpis)
kpi_df

### Interpretación de los KPIs

| KPI | Resultado | Target | Estado |
|-----|-----------|--------|--------|
| QA Score | Promedio de Evaluation_Score | >= 85% | ✅/❌ |
| Pass Rate | % evaluaciones con Status=Pass | >= 85% | ✅/❌ |
| Critical Error Rate | % evaluaciones con Critical_Error=Yes | < 2% | ✅/❌ |
| CSAT | Promedio de Customer_Satisfaction | >= 4.2 | ✅/❌ |
| Resolution Time | Promedio de minutos por caso | Tendencia | Monitor |

> **Nota:** Un QA Score alto con Pass Rate bajo indica que los scores están cerca del 85% pero muchos no alcanzan el umbral. Revisar la distribución es clave.

---
## 5. Team Performance

**Qué hacemos:** Calcular los KPIs desglosados por equipo y ordenarlos por desempeño.

**Por qué:** El promedio global oculta diferencias entre equipos. Un equipo de alto desempeño puede estar compensando a otro de bajo desempeño, creando una falsa sensación de calidad aceptable.

**Decisión de negocio:** Los equipos con QA Score bajo son candidatos para revisión de supervisión, calibración de evaluaciones o programas de capacitación focalizados.

In [ ]:
team_df = team_performance(df)
print(f'Equipos analizados: {len(team_df)}')
team_df[['Team', 'Evaluations', 'Avg_QA_Score', 'Pass_Rate',
          'Critical_Error_Rate', 'Avg_CSAT', 'Avg_Resolution_Time']]

In [ ]:
# Resumen visual del desempeño por equipo
print('Team Performance vs Targets:')
print(f'{"Team":<12} {"QA Score":<12} {"Pass Rate":<12} {"Crit.Err.":<12} {"CSAT":<8}')
print('-' * 56)
for _, row in team_df.iterrows():
    qa_icon = '✅' if row['QA_On_Target'] else '❌'
    pr_icon = '✅' if row['Pass_On_Target'] else '❌'
    ce_icon = '✅' if row['CritErr_On_Target'] else '❌'
    cs_icon = '✅' if row['CSAT_On_Target'] else '❌'
    print(f"{row['Team']:<12} {row['Avg_QA_Score']:.2f}% {qa_icon:<5} "
          f"{row['Pass_Rate']:.2f}% {pr_icon:<5} "
          f"{row['Critical_Error_Rate']:.2f}% {ce_icon:<5} "
          f"{row['Avg_CSAT']:.2f} {cs_icon}")

---
## 6. Process Performance

**Qué hacemos:** Analizar el desempeño por proceso, enfocándonos en identificar los procesos de mayor riesgo.

**Por qué:** Los errores críticos no se distribuyen uniformemente entre procesos. Algunos procesos son inherentemente más complejos o tienen protocolos más estrictos. Identificar el proceso con mayor tasa de errores críticos permite priorizar acciones de mejora.

**Decisión de negocio:** Un proceso con alta tasa de errores críticos debe revisarse primero en términos de: claridad del protocolo, frecuencia de calibración y nivel de capacitación requerido.

In [ ]:
process_df = process_performance(df)
process_df[['Process', 'Evaluations', 'Avg_QA_Score', 'Total_Errors',
            'Avg_Errors', 'Total_Critical_Errors', 'Critical_Error_Rate',
            'Avg_CSAT', 'Avg_Resolution_Time']]

In [ ]:
# Proceso con mayor riesgo
worst_process = process_df.iloc[0]
print(f"⚠️  PROCESO DE MAYOR RIESGO: {worst_process['Process']}")
print(f"   Critical Error Rate: {worst_process['Critical_Error_Rate']:.2f}% (target: < 2%)")
print(f"   Total Critical Errors: {int(worst_process['Total_Critical_Errors']):,}")
print(f"   Avg QA Score: {worst_process['Avg_QA_Score']:.2f}%")

---
## 7. Agent Performance — Top 10 & Bottom 10

**Qué hacemos:** Calcular el desempeño individual de cada agente e identificar los mejores y peores performers.

**Por qué:** El análisis a nivel agente permite acciones de coaching individualizado, reconocimiento de alto desempeño y detección temprana de agentes en riesgo.

**Por qué necesitamos mínimo de evaluaciones:**
Un agente con 5 evaluaciones y promedio 98% puede ser el mejor agente... o simplemente tuvo 5 casos fáciles en su primera semana. Con menos de 30 evaluaciones, el intervalo de confianza del promedio es demasiado amplio para tomar decisiones sobre el desempeño individual. Este umbral es una heurística conservadora, no un límite estadístico estricto.

**Decisión de negocio:**
- Top 10 → Candidatos para mentor/SME, reconocimiento, buenas prácticas compartidas
- Bottom 10 → Candidatos para plan de mejora, coaching focalizado, revisión de carga de trabajo

In [ ]:
agent_df = agent_performance(df, min_evaluations=30)
eligible_count = agent_df['Eligible_for_Ranking'].sum()
total_agents = len(agent_df)
below_target = (agent_df['QA_Below_Target'] & agent_df['Eligible_for_Ranking']).sum()

print(f'Total agentes: {total_agents}')
print(f'Elegibles para ranking (>= 30 evaluaciones): {eligible_count}')
print(f'Agentes bajo target de QA (de los elegibles): {below_target}')

In [ ]:
# TOP 10 AGENTS
top_df = top_agents(agent_df, n=10)
print('🏆 TOP 10 AGENTS por QA Score:')
top_df[['Agent', 'Team', 'Evaluations', 'Avg_QA_Score', 'Pass_Rate', 'Avg_CSAT']]

In [ ]:
# BOTTOM 10 AGENTS
bottom_df = bottom_agents(agent_df, n=10)
print('⚠️  BOTTOM 10 AGENTS por QA Score:')
bottom_df[['Agent', 'Team', 'Evaluations', 'Avg_QA_Score', 'Pass_Rate', 'Avg_CSAT']]

---
## 8. Trend Analysis

**Qué hacemos:** Analizar la evolución de los KPIs en el tiempo (diario, semanal, mensual).

**Por qué:** Los promedios globales son estáticos. El análisis de tendencia revela si la calidad está mejorando, deteriorando o estancada. Una tendencia en deterioro requiere intervención inmediata aunque el promedio global todavía esté en el target.

**Decisión de negocio:** Si el QA Score cae por debajo del target en 3 semanas consecutivas, es una señal para convocar una revisión de calibración o revisar cambios en procesos recientes.

In [ ]:
monthly_df = monthly_trend(df)
print('Tendencia mensual de KPIs:')
monthly_df[['Month_Name', 'Evaluations', 'Avg_QA_Score', 'Pass_Rate',
            'Critical_Error_Rate', 'Avg_CSAT', 'Avg_Resolution_Time']]

In [ ]:
# Comparar primer y último mes
if len(monthly_df) >= 2:
    first = monthly_df.iloc[0]
    last  = monthly_df.iloc[-1]
    diff  = last['Avg_QA_Score'] - first['Avg_QA_Score']
    direction = 'MEJORANDO ↑' if diff > 0 else 'DETERIORANDO ↓' if diff < 0 else 'ESTABLE →'
    print(f'QA Score: {first["Month_Name"]} ({first["Avg_QA_Score"]:.2f}%) → {last["Month_Name"]} ({last["Avg_QA_Score"]:.2f}%)')
    print(f'Tendencia: {direction} ({diff:+.2f} puntos porcentuales)')

---
## 9. Correlation Analysis

**Qué hacemos:** Calcular correlaciones de Pearson entre variables clave.

**Por qué:** Las correlaciones ayudan a identificar si existe una relación lineal entre variables. Esto puede apoyar hipótesis de negocio como "a mayor QA Score, mayor satisfacción del cliente".

**⚠️ ADVERTENCIA ESTADÍSTICA IMPORTANTE:**
**Correlación no implica causalidad.** Una correlación moderada entre QA Score y CSAT no significa que mejorar el score de calidad causará automáticamente mayor satisfacción. Pueden existir variables no capturadas en este dataset (tipo de cliente, complejidad del caso, canal, etc.) que expliquen parte de ambas variables.

**Decisión de negocio:** Una correlación moderada o fuerte entre QA y CSAT justifica invertir en el programa de calidad como estrategia de mejora de experiencia del cliente, aunque no garantiza resultados lineales.

In [ ]:
corr_df = correlation_analysis(df)
print('Análisis de correlaciones (Pearson r):')
print()
for _, row in corr_df.iterrows():
    strength_color = {
        'Strong': '🔴',
        'Moderate': '🟡',
        'Weak': '🟢',
        'Negligible': '⚪'
    }.get(row['Strength'], '⚪')
    print(f"  {strength_color} {row['Relationship']:<35} r = {row['Pearson_r']:>7.4f}  [{row['Strength']}]")

print()
print('⚠️  Recordatorio: Correlación ≠ Causalidad')

In [ ]:
corr_df

---
## 10. Visualizations

**Qué hacemos:** Generar los 7 gráficos profesionales del dashboard.

**Por qué:** Las visualizaciones comunican en segundos lo que las tablas tardan minutos en procesar. Un buen gráfico permite que el tomador de decisiones identifique el problema o la oportunidad sin necesidad de ser analista de datos.

**Principios de diseño aplicados:**
- Verde = on target / positivo
- Rojo = debajo del target / riesgo
- Línea roja punteada = target reference
- Títulos descriptivos (no genéricos)
- Sin 3D, sin colores decorativos

In [ ]:
# Generar y guardar todas las visualizaciones
viz_paths = generate_all_visualizations(
    df=df,
    monthly_df=monthly_df,
    team_df=team_df,
    process_df=process_df,
    top_agents_df=top_df,
    bottom_agents_df=bottom_df,
    output_dir=VIZ_DIR
)
print(f'\n{len(viz_paths)} visualizaciones generadas en: {VIZ_DIR}')

In [ ]:
# Mostrar visualizaciones en el notebook
from IPython.display import Image, display

for name, path in viz_paths.items():
    print(f'\n--- {name.upper().replace("_", " ")} ---')
    display(Image(filename=str(path), width=800))

---
## 11. Executive Insights Report

**Qué hacemos:** Generar automáticamente el reporte ejecutivo que responde las preguntas clave de negocio.

**Por qué:** Los insights transforman datos en narrativa accionable. El análisis de datos solo tiene valor cuando se convierte en decisiones. Este reporte estructura las conclusiones en:
- **FACTS:** Lo que los datos muestran objetivamente
- **INSIGHTS:** Interpretaciones apoyadas en los datos
- **RECOMMENDATIONS:** Acciones sugeridas con base en evidencia

**Principio fundamental:** No inventar explicaciones causales que los datos no permitan demostrar. Toda recomendación es una hipótesis para investigar, no una conclusión definitiva.

In [ ]:
insights = generate_executive_insights(
    df=df,
    kpis=kpis,
    team_df=team_df,
    process_df=process_df,
    agent_df=agent_df,
    monthly_df=monthly_df,
    corr_df=corr_df
)
print(insights)

---
## 12. Export to Excel

**Qué hacemos:** Guardar todos los resultados en un archivo Excel con múltiples hojas.

**Por qué:** El Excel permite que usuarios sin Python puedan acceder a los resultados del análisis, hacer filtros manuales y compartir con stakeholders que no usan herramientas de BI.

In [ ]:
from data_quality import results_to_dataframe

SUMMARY_PATH = OUTPUT_DIR / 'qa_summary.xlsx'

with pd.ExcelWriter(SUMMARY_PATH, engine='openpyxl') as writer:
    results_to_dataframe(dq_results).to_excel(writer, sheet_name='Data Quality', index=False)
    kpi_df.to_excel(writer, sheet_name='KPIs', index=False)
    team_df.to_excel(writer, sheet_name='Team Analysis', index=False)
    process_df.to_excel(writer, sheet_name='Process Analysis', index=False)
    agent_df.to_excel(writer, sheet_name='Agent Analysis', index=False)
    monthly_df.to_excel(writer, sheet_name='Monthly Trend', index=False)
    corr_df.to_excel(writer, sheet_name='Correlations', index=False)
    qa_category_distribution(df).to_excel(writer, sheet_name='QA Categories', index=False)

print(f'✅ Excel exportado: {SUMMARY_PATH}')
print(f'   Hojas incluidas: Data Quality, KPIs, Team Analysis, Process Analysis,')
print(f'   Agent Analysis, Monthly Trend, Correlations, QA Categories')

---
## Resumen Final del Análisis

### Archivos generados
| Archivo | Descripción |
|---------|-------------|
| `Output/qa_clean.csv` | Dataset limpio con columnas derivadas |
| `Output/data_quality_report.csv` | Reporte de calidad de datos |
| `Output/qa_summary.xlsx` | Excel ejecutivo con 8 hojas |
| `Output/executive_insights.txt` | Reporte ejecutivo automático |
| `visualizations/*.png` | 7 gráficos profesionales |

### Próximos pasos
1. Importar `output/qa_clean.csv` en Power BI
2. Seguir las instrucciones en `powerbi/README_POWERBI.md`
3. Implementar las medidas DAX de `powerbi/DAX_MEASURES.md`
4. Publicar el dashboard al workspace del equipo

---
*QA Analytics Project — 2026*